# Polars Bokeh Accessor Examples

Minimal examples using the `pl.DataFrame.bokeh` accessor.

In [ ]:
%load_ext autoreload
%autoreload 2

import polars as pl
import datetime as dt

from bokeh.io import output_notebook
from bokeh.plotting import show
from bokeh.plotting import figure

from spectral.charts import polars_charts
from spectral.charts.theme_manager import theme

output_notebook()
theme.set("light_minimal")

## Multiple Calls on the Same Figure

Reuse a single Bokeh figure to mimic `df.plot(ax=...)` behavior.


Figure-level and glyph-level options are passed via `figure_params={...}` and `glyph_params={...}`.


In [ ]:
from bokeh.plotting import show

df1 = pl.DataFrame(data={'x': [0, 1, 2, 3], 'A': [1, 2, 3, 4]})
df2 = pl.DataFrame(data={'x': [0, 1, 2, 3, 4], 'B1': [0.0, 0.1, 0.5, 1.5, 2], 'B2': [1, 1, 2, 3, 2]})

fig1 = df1.bokeh.plot(
    "line",
    glyph_params={"x": "x", "y": "A", "line_color": "blue", "line_alpha": 0.6},
    figure=None,
    figure_params={
        "title": "Multiple df.bokeh.plot() Calls on Same Figure",
        "width": 500,
        "height": 350,
    },
)
df1.bokeh.plot(
    "scatter",
    glyph_params={"x": "x", "y": "A", "line_color": "yellow"},
    figure=fig1,
    figure_params={"title": "title changed"},
)
df2.bokeh.plot(
    "line",
    glyph_params={"x": "x", "y": "B1", "line_color": "red"},
    figure=fig1,
    figure_params={"title": "title changed"},
)
df2.bokeh.plot(
    "line",
    glyph_params={"x": "x", "y": "B2", "line_color": "red"},
    figure=fig1,
    figure_params={"title": "title changed"},
)
df2.bokeh.plot(
    "varea",
    glyph_params={"x": "x", "y1": "B1", "y2": "B2"},
    figure=fig1,
)

show(fig1);


In [ ]:
df1 = pl.DataFrame(
    data={
        'x': [1, 2, 3, 4],
        "col0": [0, 0, 0, 0],
        'col1': [1, 2, 3, 4],
        'col2': [2, 3, 4, 5],
        'col3': [3, 4, 5, 6]
    }
)

fig1 = df1.bokeh.plot(
    "varea",
    glyph_params={"x": "x", "y1": "col0", "y2": "col1", "hatch_color": "green", "fill_color": "yellow"},
    figure=None,
    figure_params={
        "title": "Multiple df.bokeh.plot() Calls on Same Figure",
        "width": 500,
        "height": 350,
    },
)

df1.bokeh.plot(
    "varea",
    glyph_params={"x": "x", "y1": "col1", "y2": "col2"},
    figure=fig1,
    figure_params={
        "title": "Multiple df.bokeh.plot() Calls on Same Figure",
        "width": 500,
        "height": 350,
    },
)

df1.bokeh.plot(
    "varea",
    glyph_params={"x": "x", "y1": "col2", "y2": "col3", "hatch_color": "green", "fill_color": "orange", "hatch_pattern": "+"},
    figure=fig1,
    figure_params={
        "title": "Multiple df.bokeh.plot() Calls on Same Figure",
        "width": 500,
        "height": 350,
    },
)

show(fig1);


## Discover Accepted Params

Use these helpers to inspect valid keys for `figure_params` and `glyph_params`.
Both methods return a `set[str]` built from Bokeh model properties at runtime.


In [ ]:
print("Figure params:", pl.DataFrame().bokeh.accepted_figure_params())

In [ ]:
fig1 = df2.bokeh.plot(
    "vline_stack",
    glyph_params={"x": "x", "stackers": ["B1", "B2"], "line_color": "blue", "line_alpha": 0.6},
    figure=None,
    figure_params={
        "title": "Multiple df.bokeh.plot() Calls on Same Figure",
        "width": 500,
        "height": 350,
    },
)

show(fig1)

In [ ]:
fig1 = df2.bokeh.plot(
    "hbar",
    glyph_params={"y": "B1", "line_color": "blue", "line_alpha": 0.6},
    figure=None,
    figure_params={
        "title": "Multiple df.bokeh.plot() Calls on Same Figure",
        "width": 500,
        "height": 350,
    },
)

show(fig1)

## Colormap With Line + Markers

`line_color` for a single `line` glyph is scalar-only in Bokeh.
Use a constant line color, and apply `linear_cmap` to marker color instead.


In [ ]:
from bokeh.transform import linear_cmap

df_cmap = pl.DataFrame({
    "x": [0, 1, 2, 3, 4, 5],
    "y": [1.0, 1.4, 1.1, 1.9, 2.3, 2.0],
    "color_metric": [10, 25, 40, 55, 70, 90],
})

fig_cmap = df_cmap.bokeh.plot(
    "line",
    glyph_params={
        "x": "x",
        "y": "y",
        "line_width": 3,
        "line_color": "#3a3a3a",
    },
    figure_params={"title": "Line with marker colormap", "width": 600, "height": 300},
)

df_cmap.bokeh.plot(
    "scatter",
    glyph_params={
        "x": "x",
        "y": "y",
        "size": 12,
        "fill_color": linear_cmap(
            "color_metric",
            "Viridis256",
            low=df_cmap["color_metric"].min(),
            high=df_cmap["color_metric"].max(),
        ),
        "line_color": "white",
    },
    figure=fig_cmap,
)

show(fig_cmap)


## Check Transform Compatibility

Use a small probe function to test whether a given `glyph_param` accepts a transform.


In [ ]:
from bokeh.transform import linear_cmap
from bokeh.models import ColumnDataSource
from bokeh.plotting import figure

def supports_transform(glyph_name: str, param: str) -> bool:
    fig = figure(width=300, height=200)
    source = ColumnDataSource(dict(x=[0, 1], y=[1, 2], value=[10, 20]))
    transform = linear_cmap("value", "Viridis256", low=10, high=20)

    kwargs = {"x": "x", "y": "y", param: transform}
    if glyph_name == "line":
        kwargs.setdefault("line_width", 3)

    try:
        getattr(fig, glyph_name)(source=source, **kwargs)
        return True
    except Exception as exc:
        print(f"{glyph_name}.{param}: NOT supported -> {type(exc).__name__}: {exc}")
        return False

checks = [
    ("scatter", "fill_color"),
    ("scatter", "size"),
    ("line", "line_color"),
    ("line", "line_width"),
]

for glyph_name, param in checks:
    ok = supports_transform(glyph_name, param)
    if ok:
        print(f"{glyph_name}.{param}: supported")


## Transform `y` In Different Ways

Examples below compare plain `y` with transformed `y` using `dodge` and `jitter`.


In [ ]:
from bokeh.transform import dodge, jitter

df_y = pl.DataFrame({
    "x": [1, 2, 3, 4, 5, 6],
    "y": [10, 13, 12, 17, 16, 20],
})

fig_y = df_y.bokeh.plot(
    "scatter",
    glyph_params={"x": "x", "y": "y", "size": 10, "fill_color": "#1f77b4", "legend_label": "y as field"},
    figure_params={"title": "y field vs transformed y", "width": 700, "height": 320},
)

df_y.bokeh.plot(
    "scatter",
    glyph_params={
        "x": "x",
        "y": dodge("y", 1.0),
        "size": 10,
        "fill_color": "#ff7f0e",
        "legend_label": "y with dodge(+1.0)",
    },
    figure=fig_y,
)

df_y.bokeh.plot(
    "scatter",
    glyph_params={
        "x": "x",
        "y": jitter("y", width=0.35),
        "size": 10,
        "fill_color": "#2ca02c",
        "legend_label": "y with jitter(width=0.35)",
    },
    figure=fig_y,
)

fig_y.legend.location = "top_left"
fig_y.legend.click_policy = "hide"
show(fig_y)
